In [1]:
# from xopr_gline import xopr_utils
import numpy as np
import xarray as xr
import hvplot.xarray
import matplotlib.pyplot as plt
import scipy.constants
from scipy import signal
import pandas as pd
import geopandas as gpd
import xopr.opr_access
import xopr.geometry
import dask
from dask.distributed import LocalCluster
import cartopy.crs as ccrs
import geoviews.feature as gf
import time
import requests
from scipy.optimize import curve_fit
from scipy.signal import butter, filtfilt, argrelextrema

from xopr_gline.xopr_utils import extract_layer_peak_power, surface_bed_reflection_power, get_basal_layer_wgs84
from xopr_gline.empirical import erf_topography_model, get_derivatives

import dask
from dask.distributed import LocalCluster
import json
import traceback

In [2]:
cresis_fl_path = '../data/cresis_useable_flight_lines.geojson'
cresis_fl = gpd.read_file(cresis_fl_path)
cresis_fl

,fid,OBJECTID,Name,FolderPath,SymbolID,AltMode,Base,Clamped,Extruded,Snippet,PopupInfo,Shape_Leng,geometry
0,1,1,20160509_07_010,20160509_07_010.kmz,0,-1,0.0,0,0,None,None,2.977177,"MULTILINESTRING Z ((-24.39078 79.0248 0, -21.9..."
1,2,2,20160509_10_001,20160509_10_001.kmz,0,-1,0.0,0,0,None,None,2.595372,"MULTILINESTRING Z ((-19.43633 79.55342 0, -20...."
2,3,3,20160519_03_016,20160519_03_016.kmz,0,-1,0.0,0,0,None,None,1.331803,"MULTILINESTRING Z ((-30.40571 72.71069 0, -30...."
3,4,4,20160519_03_026,20160519_03_026.kmz,0,-1,0.0,0,0,None,None,1.019261,"MULTILINESTRING Z ((-30.83517 73.38396 0, -30...."
4,5,5,19950524_01_011,19950524_01_011.kmz,0,-1,0.0,0,0,None,None,2.440836,"MULTILINESTRING Z ((-47.7925 69.45962 0, -47.8..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,332,76,20140512_01_027,20140512_01_027.kmz,0,-1,0.0,0,0,None,None,1.048882,"MULTILINESTRING Z ((-66.90451 77.4937 0, -66.7..."
332,333,77,20140512_01_028,20140512_01_028.kmz,0,-1,0.0,0,0,None,None,1.175858,"MULTILINESTRING Z ((-66.62773 77.74019 0, -66...."
333,334,26,20140424_01_022,20140424_01_022.kmz,0,-1,0.0,0,0,None,None,0.779269,"MULTILINESTRING Z ((-32.60986 67.60396 0, -32...."
334,335,31,20140424_01_043,20140424_01_043.kmz,0,-1,0.0,0,0,None,None,1.090071,"MULTILINESTRING Z ((-38.18128 66.36619 0, -38...."


In [3]:
opr = xopr.opr_access.OPRConnection(cache_dir="/tmp")

In [4]:
def grab_ids(stac_item_df, frame_idx):

    # see if item is last frame
    last_index = len(stac_item_df) - 1
    
    if frame_idx == last_index:
        stac_items = stac_item_df.iloc[[frame_idx-1,frame_idx]]
        return stac_items
        
    elif frame_idx < last_index:
        stac_items = stac_item_df.iloc[[frame_idx-1,frame_idx, frame_idx+1 ]]
        return stac_items
        
    elif frame_idx > last_index:
        raise IndexError(f"Index {frame_idx} is out of range for DataFrame of length {len(stac_item_df)}")



def grounding_point_grab(reflectivity_concat, layers):

    # Get thickness
    H = layers['standard:surface']['wgs84'] - layers['standard:bottom']['wgs84']
    # Get a thickness mask so max gradient is over glacier and not melange or other noise
    H_mask = H > 200
    H_filt = H[H_mask]

    thick_slow_time_min = H_filt.slow_time.min()
    thick_slow_time_max = H_filt.slow_time.max()
    print(f'slow time thick min: {str(thick_slow_time_min.to_pandas())}\nslow time thick max: {str(thick_slow_time_max.to_pandas())}')

    
    # Get Hab
    H = layers['standard:surface']['wgs84'] - (layers['standard:bottom']['wgs84'])
    rho_sw=1024
    rho_ice=917
    # invert the bottom to make depth positive
    Hab = H - (rho_sw/rho_ice) * (layers['standard:bottom']['wgs84'] *-1)

    
    Hab_min_slowtime, Hab_max_slowtime = Hab[Hab<=100].isel(slow_time=[0,-1]).slow_time.data
    print(f'slow time Hab min: {str(Hab_min_slowtime)}\nslow time Hab max: {str(Hab_max_slowtime)}')
    
    bed_power_grad = np.gradient(reflectivity_concat['bed_power_dB']) # the Xia paper takes the second derivative but that looks worse
    reflectivity_concat['bed_power_grad'] = (('slow_time'), bed_power_grad)
    bed_power_grad_filter = reflectivity_concat['bed_power_grad'].sel(slow_time=slice(thick_slow_time_min, thick_slow_time_max) )
    bed_power_grad_filter = bed_power_grad_filter.sel(slow_time=slice(Hab_min_slowtime, Hab_max_slowtime) )
    
    # get the index of the peak bed power and grab the bed elevation index too
    grad_max_idx = bed_power_grad_filter.argmax(dim="slow_time").data
    grad_max = bed_power_grad_filter.max()
    
    grad_slowtime = bed_power_grad_filter['slow_time'][grad_max_idx]
    bed_point = layers['standard:bottom']['wgs84'].sel(slow_time=grad_slowtime.data, method='nearest')
    
    return bed_point, grad_max


In [5]:

def profile_plot(layers, bed_point, grad_max, stac_id_name):
    
    # Plot layers using elevation data and slow_time
    fig, (ax1, ax2, ax3) = plt.subplots(nrows=3, ncols=1, figsize=(8,6), sharex=True)
    fig.tight_layout(pad=4.0)
    

    layers['standard:surface']['wgs84'].plot(ax=ax1, x='along_track', linewidth=1, linestyle=':')
    layers['standard:bottom']['wgs84'].plot(ax=ax1, x='along_track', linewidth=1, linestyle=':')
    
    ax1.scatter(bed_point.along_track, bed_point, color='r', s=20, label="Grounding Point")
    ax1.set_title(f'{stac_id_name} Grounding Point')
    
    # ax1.axvspan(along_track_lat1.data, along_track_lat2.data, color='tab:green',
    #             alpha=0.5, label='2011-2015 gz')
    
    
    reflectivity_concat['bed_power_dB'].plot(ax=ax2, x='along_track', linewidth=1)
    # for result in results:
    #     result = xopr.radar_util.add_along_track(result)
    #     result['bed_power_dB'].plot(ax=ax2, x='along_track', label='Bed Power dB', color='tab:green')
    # ax3.scatter(grad_slowtime, grad_max, color='r', s=20, label="Grounding Point")
    
    
    # Plot layers using elevation data
    #ax3 plot
    reflectivity_concat['bed_power_grad'].plot(ax=ax3, x='along_track', label='bed_grad', color='tab:green')
    ax3.scatter(bed_point.along_track, grad_max, color='r', s=20, label="Grounding Point")
    
    ax1.legend()
    # ax3.legend()
    ax3.set_title('Bed power gradient')
    fig.savefig(f'/home/m484s199/gline_figures/{stac_id_name}.png', dpi=300)
    plt.close()

In [6]:
# Select a segment
for name in cresis_fl["Name"][:10]:

    print(f'Processing {name}')
    
    selected_segment = name[:-4]
    print(f"Selected segment: {selected_segment}")
    
    # Query frames
    stac_items = opr.query_frames(
                    segment_paths=[selected_segment]
                    )
    
    print(f"Found {len(stac_items)} frames")
    
    stac_id = f'Data_{name}'
    frame_idx = stac_items.index.get_loc(stac_id)

    stac_items = grab_ids(stac_items, frame_idx)


    # Load the radar data
    frames = opr.load_frames(stac_items)
    print(f"Loaded {len(frames)} frames")


    # Merge frames into a single flight line
    # If using a single stac item id make sure to put frames in a list
    flight_line = xopr.merge_frames(frames)

    # Load layers for the merged flight line
    layers = opr.get_layers(flight_line)
    print(f"Available layers: {list(layers.keys())}")



    for layer_idx in layers:
        layers[layer_idx] = xopr.radar_util.add_along_track(layers[layer_idx])
        layers[layer_idx] = xopr.layer_twtt_to_range(layers[layer_idx], layers["standard:surface"], vertical_coordinate='wgs84')
        layers[layer_idx] = xopr.layer_twtt_to_range(layers[layer_idx], layers["standard:surface"], vertical_coordinate='range')

    flight_line = xopr.radar_util.add_along_track(flight_line)
    flight_line = xopr.radar_util.interpolate_to_vertical_grid(flight_line, vertical_coordinate='wgs84')


    client = LocalCluster().get_client()


    stac_list = [row for _, row in stac_items.iterrows()]
    futures = client.map(surface_bed_reflection_power, stac_list, opr=opr)

    # Process results as they complete, capturing exceptions
    results = []
    for future in dask.distributed.as_completed(futures):
        try:
            result = future.result()
            results.append(result)
        except Exception as e:
            print(traceback.format_exc())

    try:
        reflectivity_concat = xr.concat(results, dim="slow_time").sortby("slow_time")
        reflectivity_concat = xopr.radar_util.add_along_track(reflectivity_concat)
        
    except Exception as e:
        print(traceback.format_exc())
        
    failed_items = []
    try:
        bed_point, grad_max = grounding_point_grab(reflectivity_concat, layers)
        
        # plot profile and bed pick
        profile_plot(layers, bed_point, grad_max, stac_id)
    except Exception as e:
        # Save both the item and the string version of the error
        failed_items.append({"item": name, "error": str(e)})


# Save the failed items to a file for later
with open("/home/m484s199/gline_figures/log/failed_items.json", "w") as f:
    json.dump(failed_items, f, indent=4)

Processing 20160509_07_010
Selected segment: 20160509_07
Found 10 frames
Loaded 2 frames
Available layers: ['standard:surface', 'standard:bottom']


/home/m484s199/xopr-gline/.venv/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 40645 instead
  warnings.warn(
/home/m484s199/xopr-gline/src/xopr_gline/xopr_utils.py:73: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'slow_time' ('slow_time',) The recommendation is to set join explicitly for this case.
  reflectivity_dataset = xr.merge([
/home/m484s199/xopr-gline/src/xopr_gline/xopr_utils.py:81: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes ar

slow time thick min: 2016-05-09T13:19:40.960054159
slow time thick max: 2016-05-09T13:29:29.371404648
slow time Hab min: 2016-05-09T13:27:32.067432404
slow time Hab max: 2016-05-09T13:32:39.990003824
Processing 20160509_10_001
Selected segment: 20160509_10
Found 1 frames
Loaded 2 frames
Available layers: ['standard:surface', 'standard:bottom', ':layer_01', ':layer_02']


/home/m484s199/xopr-gline/.venv/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42307 instead
  warnings.warn(
/home/m484s199/xopr-gline/src/xopr_gline/xopr_utils.py:81: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'slow_time' ('slow_time',) The recommendation is to set join explicitly for this case.
  reflectivity_dataset = xr.merge([reflectivity_dataset, flight_line_metadata])


slow time thick min: 2016-05-09T13:48:03.182053566
slow time thick max: 2016-05-09T13:54:39.952481270
slow time Hab min: 2016-05-09T13:46:57.934937954
slow time Hab max: 2016-05-09T13:54:39.952481270
Processing 20160519_03_016
Selected segment: 20160519_03
Found 28 frames
Loaded 3 frames
Available layers: ['standard:surface', 'standard:bottom']


...

Traceback (most recent call last):
  File "/tmp/ipykernel_348208/936738462.py", line 56, in <module>
    result = future.result()
             ^^^^^^^^^^^^^^^
  File "/home/m484s199/xopr-gline/.venv/lib/python3.11/site-packages/distributed/client.py", line 406, in result
    return self.client.sync(self._result, callback_timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/m484s199/xopr-gline/src/xopr_gline/xopr_utils.py", line 68, in surface_bed_reflection_power
    surface_repicked_twtt, surface_power = extract_layer_peak_power(frame, layers["standard:surface"]['twtt'], layer_selection_margin_twtt)
      ^^^^^^^^^^^^^^^^^
KeyError: 'standard:surface'



2026-04-03 16:10:06,563 - distributed.worker - ERROR - Compute Failed
Key:       surface_bed_reflection_power-ebb45662da28e0cfef99acb5271983eb
State:     executing
Task:  <Task 'surface_bed_reflection_power-ebb45662da28e0cfef99acb5271983eb' surface_bed_reflection_power(..., ...)>
Exception: "KeyError('standard:surface')"
Traceback: '  File "/home/m484s199/xopr-gline/src/xopr_gline/xopr_utils.py", line 68, in surface_bed_reflection_power\n    surface_repicked_twtt, surface_power = extract_layer_peak_power(frame, layers["standard:surface"][\'twtt\'], layer_selection_margin_twtt)\n                                                                           ~~~~~~^^^^^^^^^^^^^^^^^^^^\n'

2026-04-03 16:10:06,757 - distributed.worker - ERROR - Compute Failed
Key:       surface_bed_reflection_power-c4b2a9953097c318e4e799b25a21d720
State:     executing
Task:  <Task 'surface_bed_reflection_power-c4b2a9953097c318e4e799b25a21d720' surface_bed_reflection_power(..., ...)>
Exception: "KeyError('standa

Traceback (most recent call last):
  File "/tmp/ipykernel_348208/936738462.py", line 56, in <module>
    result = future.result()
             ^^^^^^^^^^^^^^^
  File "/home/m484s199/xopr-gline/.venv/lib/python3.11/site-packages/distributed/client.py", line 406, in result
    return self.client.sync(self._result, callback_timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/m484s199/xopr-gline/src/xopr_gline/xopr_utils.py", line 68, in surface_bed_reflection_power
    surface_repicked_twtt, surface_power = extract_layer_peak_power(frame, layers["standard:surface"]['twtt'], layer_selection_margin_twtt)
      ^^^^^^^^^^^^^^^^^
KeyError: 'standard:surface'

Traceback (most recent call last):
  File "/tmp/ipykernel_348208/936738462.py", line 56, in <module>
    result = future.result()
             ^^^^^^^^^^^^^^^
  File "/home/m484s199/xopr-gline/.venv/lib/python3.11/site-packages/distributed/client.py", line 406, in result
    return self.clie

/home/m484s199/xopr-gline/.venv/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45569 instead
  warnings.warn(
/home/m484s199/xopr-gline/src/xopr_gline/xopr_utils.py:73: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'slow_time' ('slow_time',) The recommendation is to set join explicitly for this case.
  reflectivity_dataset = xr.merge([
/home/m484s199/xopr-gline/src/xopr_gline/xopr_utils.py:73: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes ar

slow time thick min: 2002-05-30T13:07:43.634852052
slow time thick max: 2002-05-30T13:13:59.951398969
slow time Hab min: 2002-05-30T13:07:43.634852052
slow time Hab max: 2002-05-30T13:18:42.994089007
